# Assignment 4

Deadline: 13.05.2026, 12:00 CET

- Gallo Alessandro , 25-732-140 , alessandro.gallo2@uzh.ch
- Maruccio Anna , 25-742-800 , anna.maruccio@uzh.ch
- Perbellini Cesare, 25-741-257, cesare.perbellini@uzh.ch
- Venturi Matilde , 25-741-059 , matilde.venturi@uzh.ch

In [5]:

# Standard library imports
import os
import sys

# Third party imports
import numpy as np
import pandas as pd
import statsmodels.api as sm    # for regression analysis in 1.c) (or any other regression library you prefer)


project_root = os.path.dirname(os.path.dirname(os.getcwd()))   # Change this path if needed
src_path = os.path.join(project_root, 'qpmwp-course', 'src') #changed for Macbook
sys.path.append(project_root)
sys.path.append(src_path)


# Local modules imports
from helper_functions import (
    load_pickle,
    load_data_spi,
    align_market_data_with_jkp_data,
)
from estimation.covariance import Covariance
from optimization.optimization import PercentilePortfolio
from backtesting.backtest_item_builder.bib_classes import (
    SelectionItemBuilder,
    OptimizationItemBuilder,
)
from backtesting.backtest_item_builder.bibfn_selection import (
    bibfn_selection_gaps,
    bibfn_selection_min_volume,
    bibfn_selection_jkp_data_scores,
)
from backtesting.backtest_item_builder.bibfn_optimization_data import (
    bibfn_scores,
)


from backtesting.backtest_data import BacktestData
from backtesting.backtest_service import BacktestService
from backtesting.backtest import Backtest


/Users/matildeventuri/Desktop/Documents/GitHub/qpmwp-course/src/estimation/black_litterman.py:43: SyntaxWarning: invalid escape sequence '\S'
  Prior covariance matrix \\( \Sigma_{\text{prior}} \\).


## Constants

In [ ]:


PATH_TO_DATA =    # <change this to your path to data>
SAVE_PATH =       # <change this to your path where you want to store the backtest>

NameError: name 'Path' is not defined

## Load data and initialize BacktestData class
- market data (from parquet file)
- jkp data (from parquet file)
- swiss performance index, SPI (from csv file)

In [ ]:
# Load market and jkp data from parquet files
market_data = pd.read_parquet(path = f'{PATH_TO_DATA}market_data.parquet')
jkp_data = pd.read_parquet(path = f'{PATH_TO_DATA}jkp_data.parquet')
spi = load_data_spi(path='../data/')

# Align market data with jkp data
market_data_ffill, jkp_data = align_market_data_with_jkp_data(market_data=market_data, jkp_data=jkp_data)

# Instantiate the BacktestData class
# and set the market, jkp, and benchmark data as attributes
data = BacktestData()
data.market_data = market_data_ffill  # notice that we use the forward filled market data here
data.jkp_data = jkp_data
data.bm_series = spi

## Define a grid or rebalancing dates

In [ ]:
n_month = 3 # We want to rebalance every n_month months
jkp_data_dates = (
    jkp_data
    .index.get_level_values('date')
    .unique().sort_values()
)
rebdates = (
    jkp_data_dates[
        jkp_data_dates > market_data.index.get_level_values('date').min()
    ][::n_month]
    .strftime('%Y-%m-%d').tolist()
)
rebdates = [date for date in rebdates if date > '2002-01-01']
rebdates

## 1. a) Define the key data fields that characterize a factor theme

- Beside the pre-defined fields, choose three factor themes from table 9 of the Global Factor Data Documentation (See https://jkpfactors-data.s3.amazonaws.com/documents/Documentation.pdf, Section 10.) and add the individual fields.

**(2 points)**

In [ ]:
#guardare da pag 38 

JKP_FIELDS_QUALITY = [
    'at_turnover',
    'cop_at',
    'cop_atl1',
    'dgp_dsale',
    'gp_at',
    'gp_atl1',
    'mispricing_perf',
    'ni_inc8q',
    'niq_at',
    'op_at',
    'op_atl1',
    'opex_at',
    'qmj_prof',
    'qmj_growth',
    'qmj_safety',
    'sale_bev',
]

JKP_FIELDS_VALUE = [
    'at_me',
    'be_me',
    'bev_mev',
    'debt_me',
    'div12m_me',
    'ebitda_mev',
    'eq_dur',
    'eqnetis_at',
    'eqnpo_12m',
    'eqnpo_me',
    'fcf_me',
    'ival_me',
    'netis_at',
    'ni_me',
    'ocf_me',
    'sale_me',
]

JKP_FIELDS_MOMENTUM = [
    'prc_highprc_252d',
    'resff3_6_1',
    'resff3_12_1',
    'ret_3_1',
    'ret_6_1',
    'ret_9_1',
    'ret_12_1',
    'seas_1_1na',
]

# <add the name of the factor theme here>
JKP_FIELDS_INVESTMENTS= [
    'aliq_at',
    'at_gr1',
    'be_gr1a',
    'capx_gr1',
    'capx_gr2',
    'capx_gr3',
    'coa_gr1a',
    'col_gr1a',
    'emp_gr1',
    'inv_gr1',
    'inv_gr1a',
    'lnoa_gr1a',
    'mispricing_mgmt',
    'ncoa_gr1a',
    'nncoa_gr1a',
    'noa_gr1a',
    'ppeinv_gr1a',
    'ret_60_12',
    'sale_gr1',
    'sale_gr3',
    'saleq_gr1',
    'seas_2_5na',
]

# <add the name of the factor theme here>
JKP_FIELDS_SIZE = [
    'ami_126d',
    'dolvol_126d',
    'market_equity',
    'prc',
    'rd_me',
]

# <add the name of the factor theme here>
JKP_FIELDS_PROFITABILITY = [
    'dolvol_var_126d',
    'ebit_bev',
    'ebit_sale',
    'f_score',
    'ni_be',
    'niq_be',
    'o_score',
    'ocf_at',
    'ope_be',
    'ope_bel1',
    'turnover_var_126d',
]


JKP_FIELDS = {
    'quality': JKP_FIELDS_QUALITY,
    'value': JKP_FIELDS_VALUE,
    'momentum': JKP_FIELDS_MOMENTUM,
    'inv': JKP_FIELDS_INVESTMENTS,
    'size': JKP_FIELDS_SIZE,
    'profit': JKP_FIELDS_PROFITABILITY,
}

## 1. b) Factor series for each factor theme.

- For each factor theme, run a backtest for the top‑quintile portfolio and a second backtest for the bottom‑quintile portfolio.

- Simulate the return streams for both backtests and define the factor series as a long-short portfolio going long the top quintile portfolio and going short the bottom quintile portfolio.

- Plot the cumulative returns of the top quintile portfolio, the bottom quintile portfolio, the long-short factor portfolio, and the benchmark series.

**(8 points)**

In [ ]:
return_series = data.get_return_series()

factor_results = {}

for factor_theme, factor_fields in JKP_FIELDS.items():
    print(f'Running theme: {factor_theme}')

    selection_item_builders = {
        'gaps': SelectionItemBuilder(
            bibfn=bibfn_selection_gaps,
            width=365 * 3,
            n_days=10,
        ),
        'min_volume': SelectionItemBuilder(
            bibfn=bibfn_selection_min_volume,
            width=365,
            min_volume=500_000,
            agg_fn=np.median,
        ),
        'jkp_data_scores': SelectionItemBuilder(
            bibfn=bibfn_selection_jkp_data_scores,
            fields=factor_fields,
        ),
    }

    # ✅ SOLO SCORES
    optimization_item_builders = {
        'scores': OptimizationItemBuilder(
            bibfn=bibfn_scores,
            fields=factor_fields,
        ),
    }

    bs = BacktestService(
        data=data,
        selection_item_builders=selection_item_builders,
        optimization_item_builders=optimization_item_builders,
        rebdates=rebdates,
        quiet=True,
    )

    # ---------------- TOP QUINTILE ----------------
    bs.optimization = PercentilePortfolio(
        percentile=80,
        sign='>=',
    )
    bt_top = Backtest()
    bt_top.run(bs=bs)

    # ---------------- BOTTOM QUINTILE ----------------
    bs.optimization = PercentilePortfolio(
        percentile=20,
        sign='<=',
    )
    bt_bottom = Backtest()
    bt_bottom.run(bs=bs)

    # ---------------- SIMULATION ----------------
    sim_top = bt_top.strategy.simulate(
        return_series=return_series,
        fc=0,
        vc=0,
    ).dropna()

    sim_bottom = bt_bottom.strategy.simulate(
        return_series=return_series,
        fc=0,
        vc=0,
    ).dropna()

    sim_df = pd.concat({
        'Top Quintile': sim_top,
        'Bottom Quintile': sim_bottom,
        'Benchmark': data.bm_series,
    }, axis=1).dropna()

    # Long-short factor
    sim_df['Long-Short Factor'] = (
        sim_df['Top Quintile'] - sim_df['Bottom Quintile']
    )

    factor_results[factor_theme] = sim_df

    # Plot
    np.log(1 + sim_df).cumsum().plot(
        figsize=(12, 7),
        title=f'Cumulative Returns - {factor_theme}'
    )

## 1. c) Factor analysis

- First, compute a factor-mix return series by averaging the returns of the top-quintile portfolio simulations that you have computed above. 

- Second, run an ordinary least squares regression of y on X, where y is your factor-mix series and X contains i) a constant, ii) the SPI return series, and iii) the factor return series computed in 1.b).

- Use monthly returns.

- Print a summary table of the regression output

**(5 points)**

In [ ]:
# <your code here>